In [1]:
!pip install pyroomacoustics
from google.colab import drive
import pyroomacoustics as pra
import pandas as pd
import numpy as np
import torchaudio
import zipfile
import ast
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.7 MB/s eta 0:00:00


In [2]:
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/dataset.zip"
extract_path = "/content/dataset/"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP başarıyla çıkarıldı")

test_csv = '/content/drive/MyDrive/SSL_Projesi/test_metadata.csv'
df = pd.read_csv(test_csv)

print(df.head())
print("Toplam test örneği:", len(df))

Mounted at /content/drive
ZIP başarıyla çıkarıldı
                                          audio_path  azimuth_deg  distance_m  \
0  /content/dataset/dataset/session_1775030812_5c...    79.932860    2.810450   
1  /content/dataset/dataset/session_1775040605_e2...   262.116227    4.355966   
2  /content/dataset/dataset/session_1775040766_d2...    70.257374    3.220559   
3  /content/dataset/dataset/session_1775030073_23...   234.963847    3.535452   
4  /content/dataset/dataset/session_1775041687_e1...   259.779367    2.567600   

      pos_x     pos_y  pos_z  snr_db  ambient_noise_db  rt60 room_dimension  
0  2.991272  4.900000    1.5      30                15   0.3      (5, 5, 3)  
1  4.402518  0.685205    2.5       0                 0   0.3    (10, 10, 5)  
2  3.587891  4.900000    1.5      30                 0   0.5      (5, 5, 3)  
3  0.470321  0.100000    1.5      15                30   0.7      (5, 5, 3)  
4  2.044407  0.100000    1.5       0                 0   0.7      (5, 5, 

In [3]:
radius = 0.1

mics = []

for i in range(7):
    angle = 2 * np.pi * i / 7
    x = radius * np.cos(angle)
    y = radius * np.sin(angle)
    mics.append([x, y, 0])

mics.append([0, 0, 0])

mic_locs = np.array(mics).T

In [4]:
def music(signals, fs, nfft, mic_locs):

    L = nfft
    hop = nfft // 2

    X = pra.transform.stft.analysis(
        signals.T,
        L,
        hop
    )

    X = X.transpose(2, 1, 0)

    doa = pra.doa.MUSIC(
        mic_locs,
        fs=fs,
        nfft=nfft,
        num_src=1
    )

    doa.locate_sources(X)
    # 1. Tahmin
    estimated_angle = float(np.degrees(doa.azimuth_recon[0]) % 360)

    # 2. Çizim için Spektrum ve Açı verilerini yakala
    spatial_spectrum = doa.grid.values
    grid_angles = np.degrees(doa.grid.azimuth)

    return estimated_angle, spatial_spectrum, grid_angles

In [5]:
print(df.columns)

Index(['audio_path', 'azimuth_deg', 'distance_m', 'pos_x', 'pos_y', 'pos_z',
       'snr_db', 'ambient_noise_db', 'rt60', 'room_dimension'],
      dtype='object')


In [6]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

def save_combined_spectrum_plot(
    grid_angles,
    spec_success, true_success, est_success,
    spec_fail, true_fail, est_fail,
    filename="/content/music_karsilastirma.png"
):
    # 1. MAKALE STANDARTLARI İÇİN SEABORN AYARLARI
    sns.set_theme(style="ticks", context="paper", font_scale=1.2)
    plt.rcParams["font.family"] = "serif"

# 1 satır, 2 sütunlu geniş bir figür oluştur
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

   # --- Yeni Akademik Renk Paleti ---
    color_spectrum = "#003366"  # Tok Akademik Lacivert
    color_true = "#d7191c"      # Dikkat çekici Kırmızı
    color_est = "#000000"       # Siyah (Kesik çizgilerde en iyi kontrast)

    # ==========================================
    # (a) BAŞARILI SENARYO (Sol Grafik)
    # ==========================================
    ax1 = axes[0]
    spec_success_norm = spec_success / np.max(spec_success)

    sns.lineplot(x=grid_angles, y=spec_success_norm, ax=ax1,
                label="Psödo-Spektrum $P(\\theta)$", color=color_spectrum, linewidth=2)
    ax1.axvline(x=true_success, color=color_true, linestyle='--', linewidth=2.5,
                label=f"Gerçek Açı ({true_success:.1f}°)")
    ax1.axvline(x=est_success, color=color_est, linestyle=':', linewidth=2.5,
                label=f"Tahmin ({est_success:.1f}°)")

    ax1.set_xlim([0, 360])
    ax1.set_ylim([0, 1.05])
    ax1.set_xlabel("Arama Açısı (Derece)", fontweight='bold')
    ax1.set_ylabel("Normalize Psödo-Spektrum", fontweight='bold')
    ax1.set_title("(a) Başarılı Kestirim", pad=15, fontweight='bold')

    ax1.legend(loc="lower right", frameon=True, edgecolor='black')
    ax1.grid(axis='y', linestyle=':', alpha=0.6)

    # ==========================================
    # (b) HATALI SENARYO (Sağ Grafik)
    # ==========================================
    ax2 = axes[1]
    spec_fail_norm = spec_fail / np.max(spec_fail)

    sns.lineplot(x=grid_angles, y=spec_fail_norm, ax=ax2,
                label="Psödo-Spektrum $P(\\theta)$", color=color_spectrum, linewidth=2)
    ax2.axvline(x=true_fail, color=color_true, linestyle='--', linewidth=2.5,
                label=f"Gerçek Açı ({true_fail:.1f}°)")
    ax2.axvline(x=est_fail, color=color_est, linestyle=':', linewidth=2.5,
                label=f"Tahmin ({est_fail:.1f}°)")

    ax2.set_xlim([0, 360])
    ax2.set_ylim([0, 1.05])
    ax2.set_xlabel("Arama Açısı (Derece)", fontweight='bold')
    ax2.set_ylabel("Normalize Psödo-Spektrum", fontweight='bold')
    ax2.set_title("(b) Hatalı Kestirim", pad=15, fontweight='bold')

    ax2.legend(loc="lower right", frameon=True, edgecolor='black')
    ax2.grid(axis='y', linestyle=':', alpha=0.6)

    # ==========================================
    # SON RÖTUŞLAR VE KAYDETME
    # ==========================================
    sns.despine(fig=fig, top=True, right=True)
    plt.tight_layout()

    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.close()
    print(f"✅ Karşılaştırmalı grafik başarıyla kaydedildi: {filename}")

In [8]:
# ==========================================
# GÜVENLİ İŞLEM DÖNGÜSÜ (MUSIC)
# ==========================================
results = []
plot_data = []
nfft = 1024

# Sütun isimlerini dinamik kontrol et
col_file = "audio_path" if "audio_path" in df.columns else "file"
col_true = "azimuth_deg" if "azimuth_deg" in df.columns else "true_angle"

# Test dosyalarını filtrele
hedef_kelimeler = r"1775032012_e70359/sample_00002\.wav|1775030993_77e7d1/sample_00002\.wav"
test_df = df[df[col_file].str.contains(hedef_kelimeler, na=False, regex=True)].reset_index(drop=True)
records = test_df.to_dict(orient="records")

for i, row in enumerate(records):
    rel_path = row[col_file]
    true_angle = row[col_true]
    wav_path = os.path.join(extract_path, rel_path)

    # Oda boyutu ve mikrofon Z ekseni
    mic_locs[2, :] = ast.literal_eval(row["room_dimension"])[2] / 2

    waveform, sr = torchaudio.load(wav_path)
    waveform = waveform.numpy()

    # MUSIC Çağrısı (Artık 3 değer dönüyor)
    estimated_angle, spatial_spectrum, grid_angles = music(waveform, sr, nfft, mic_locs)

    # Hata hesapla
    raw_error = abs(true_angle - estimated_angle)
    error = min(raw_error, 360 - raw_error)

    results.append({"method": "music", "file": rel_path, "true_angle": true_angle, "estimated_angle": estimated_angle, "angular_error": error})
    plot_data.append({"spec": spatial_spectrum, "true": true_angle, "est": estimated_angle, "error": error})

    print(f"[{i+1}/2] İşlendi: {rel_path.split('/')[-1]} | Hata: {error:.2f}°")

# ==========================================
# GRAFİK TETİKLEME
# ==========================================
if len(plot_data) == 2:
    idx_success, idx_fail = (0, 1) if plot_data[0]["error"] < plot_data[1]["error"] else (1, 0)

    # Senin `save_combined_spectrum_plot` fonksiyonunu çağırıyoruz
    save_combined_spectrum_plot(
        grid_angles=grid_angles,
        spec_success=plot_data[idx_success]["spec"],
        true_success=plot_data[idx_success]["true"],
        est_success=plot_data[idx_success]["est"],
        spec_fail=plot_data[idx_fail]["spec"],
        true_fail=plot_data[idx_fail]["true"],
        est_fail=plot_data[idx_fail]["est"],
        filename="/content/music_karsilastirma.png"
    )

results_df = pd.DataFrame(results)
results_df.to_csv("music-results.csv", index=False)

[1/2] İşlendi: sample_00002.wav | Hata: 172.46°
[2/2] İşlendi: sample_00002.wav | Hata: 0.00°
✅ Karşılaştırmalı grafik başarıyla kaydedildi: /content/music_karsilastirma.png
